# Task 3 · Cleaning Data — Retail Sales Dataset

**Objective:** turn a deliberately messy retail sales file into a clean, analysis-ready dataset and document every decision.

**Tech stack:** Python, pandas, numpy, Jupyter Notebook

**How this notebook is organised**

| # | Section | Checklist item |
|---|---------|----------------|
| 1 | Load + data quality report | Data quality report |
| 2 | Empty rows and duplicates | Duplicate removal |
| 3 | Standardise text and dates | Standardisation |
| 4 | Fix impossible values | Value range anomalies |
| 5 | Outlier detection | Outlier detection |
| 6 | Missing data | Missing data handling |
| 7 | Data types | Data type correction |
| 8 | Before vs. after | Summary table |
| 9 | Save the clean file | Save to CSV |

**Why this order?** Duplicates, blank rows, inconsistent labels and impossible values all distort the medians and modes used to fill gaps. So I fix what is *wrong* first (sections 2 to 5) and fill what is *missing* afterwards (section 6).

Every change is also written to a `cleaning_log` that is shown at the end.

## 1. Load the data and build the data quality report
`raw` is never modified. All cleaning happens on a copy called `df`, so the before/after comparison in section 8 is honest.

In [23]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

RAW_PATH = "retail_sales_dataset.csv"
CLEAN_PATH = "retail_sales_cleaned.csv"

raw = pd.read_csv(RAW_PATH)
df = raw.copy()

# Every cleaning decision gets recorded here and is displayed at the end
cleaning_log = []
def log(step, action, affected, why=""):
    cleaning_log.append({"step": step, "action": action, "affected": affected, "why": why})

print(f"Loaded {raw.shape[0]:,} rows x {raw.shape[1]} columns")
display(raw.head())

Loaded 4,310 rows x 21 columns


,order_id,order_date,customer_id,customer_name,age,gender,region,city,product_category,product_name,quantity,unit_price,discount_pct,sales_amount,profit,shipping_cost,payment_method,customer_satisfaction,return_flag,order_status,days_to_ship
0,ORD-03903,2024-08-15,CUST32603,Aarav Mehta,39.0,Other,Central,Raipur,Groceries,Sugar,8.0,346.32,NaN,4043.14,289.18,56.04,Debit Card,NaN,True,Returned,7.0
1,ORD-02198,2022-08-16,CUST07881,Sanjay Verma,28.0,Male,West,Mumbai,Beauty,Sunscreen,8.0,858.66,0.08,9147.31,5355.56,64.49,Debit Card,5.0,False,Delivered,2.0
2,ORD-01348,2021-08-29,CUST49296,Pooja Sharma,41.0,Other,West,Surat,Furniture,Sofa,4.0,25941.32,0.14,110156.97,36956.51,67.65,Debit Card,4.0,False,Delivered,5.0
3,ORD-02076,2022-07-05,CUST05587,Priya Reddy,29.0,Other,West,Ahmedabad,Furniture,Bed Frame,9.0,22071.92,0.17,262066.11,66644.99,54.47,EMI,3.0,False,Delivered,4.0
4,ORD-00287,2020-05-07,CUST58035,Ritu Sharma,9.0,Male,Central,Nagpur,Groceries,Cooking Oil,1.0,320.06,0.31,323.95,31.38,48.81,Net Banking,4.0,False,Pending,4.0


In [24]:
# ---- Reference definitions used by the report and by the cleaning steps ----

# The dtype each column SHOULD have
EXPECTED_DTYPES = {
    "order_id": "string",   "order_date": "datetime", "customer_id": "string", "customer_name": "string",
    "age": "integer",       "gender": "string",       "region": "string",      "city": "string",
    "product_category": "string", "product_name": "string", "quantity": "integer",
    "unit_price": "float",  "discount_pct": "float",  "sales_amount": "float", "profit": "float",
    "shipping_cost": "float", "payment_method": "string", "customer_satisfaction": "integer",
    "return_flag": "bool",  "order_status": "string", "days_to_ship": "integer",
}
DTYPE_CHECKS = {
    "string": pd.api.types.is_string_dtype,
    "datetime": pd.api.types.is_datetime64_any_dtype,
    "integer": pd.api.types.is_integer_dtype,
    "float": pd.api.types.is_float_dtype,
    "bool": pd.api.types.is_bool_dtype,
}

# Business rules: values outside (min, max) are impossible. None = no limit on that side.
RANGE_RULES = {
    "age": (1, 100),
    "quantity": (1, 100),
    "unit_price": (0, None),
    "sales_amount": (0, None),
    "shipping_cost": (0, None),
    "days_to_ship": (0, None),
    "discount_pct": (0, 1),
    "customer_satisfaction": (1, 5),
}

def dtype_report(frame):
    rows = []
    for col, kind in EXPECTED_DTYPES.items():
        rows.append({"column": col, "current_dtype": str(frame[col].dtype),
                     "expected": kind, "correct": bool(DTYPE_CHECKS[kind](frame[col]))})
    return pd.DataFrame(rows)

def out_of_range(frame, col):
    lo, hi = RANGE_RULES[col]
    bad = pd.Series(False, index=frame.index)
    if lo is not None:
        bad |= frame[col] < lo
    if hi is not None:
        bad |= frame[col] > hi
    return bad

def snapshot(frame):
    d = dtype_report(frame)
    nulls = frame[list(EXPECTED_DTYPES)].isna().sum()
    return {
        "Row count": len(frame),
        "Null cells (total)": int(nulls.sum()),
        "Columns containing nulls": int((nulls > 0).sum()),
        "Duplicate rows": int(frame.duplicated().sum()),
        "Columns with correct dtype": f"{int(d['correct'].sum())} / {len(d)}",
        "Dtype accuracy": f"{d['correct'].mean():.0%}",
    }

before = snapshot(raw)

### 1a. Missing values per column

In [25]:
nulls = raw.isna().sum()
null_report = pd.DataFrame({"nulls": nulls, "pct_of_rows": (nulls / len(raw) * 100).round(2)})
display(null_report.sort_values("nulls", ascending=False))

empty_rows = int(raw.isna().all(axis=1).sum())
print(f"\nRows where EVERY column is empty: {empty_rows}")

,nulls,pct_of_rows
customer_satisfaction,378,8.77
discount_pct,167,3.87
age,160,3.71
quantity,140,3.25
days_to_ship,130,3.02
gender,30,0.70
region,30,0.70
city,30,0.70
product_category,30,0.70
product_name,30,0.70



Rows where EVERY column is empty: 30


### 1b. Duplicate rows

In [26]:
dup_total = int(raw.duplicated().sum())
dup_non_empty = int(raw.dropna(how="all").duplicated().sum())
print(f"Duplicate rows (raw file):            {dup_total}")
print(f"  of which are repeats of blank rows: {dup_total - dup_non_empty}")
print(f"  real duplicate records:             {dup_non_empty}")

Duplicate rows (raw file):            109
  of which are repeats of blank rows: 29
  real duplicate records:             80


### 1c. Data type issues
A column is flagged when its current dtype is not the dtype the data needs (dates as `datetime`, whole numbers as integers, flags as `bool`, and so on).

In [27]:
dtype_before = dtype_report(raw)
display(dtype_before[~dtype_before["correct"]])

,column,current_dtype,expected,correct
1,order_date,str,datetime,False
4,age,float64,integer,False
10,quantity,float64,integer,False
17,customer_satisfaction,float64,integer,False
18,return_flag,object,bool,False
20,days_to_ship,float64,integer,False


### 1d. Value range anomalies
Each rule below is a plain business sanity check (for example, nobody is 999 years old and a quantity cannot be negative).

In [28]:
rows = []
for col, (lo, hi) in RANGE_RULES.items():
    rows.append({"column": col, "allowed": f"{lo} to {hi if hi is not None else 'no limit'}",
                 "min_in_data": raw[col].min(), "max_in_data": raw[col].max(),
                 "out_of_range": int(out_of_range(raw, col).sum())})
range_report = pd.DataFrame(rows)
display(range_report)

,column,allowed,min_in_data,max_in_data,out_of_range
0,age,1 to 100,-7.00,999.00,26
1,quantity,1 to 100,-1.00,999.00,26
2,unit_price,0 to no limit,50.26,79957.92,0
3,sales_amount,0 to no limit,39.45,17360314.17,0
4,shipping_cost,0 to no limit,-4.29,133.35,2
5,days_to_ship,0 to no limit,-1.00,100.00,5
6,discount_pct,0 to 1,0.00,0.40,0
7,customer_satisfaction,1 to 5,1.00,5.00,0


### 1e. Inconsistent formatting

In [29]:
print("gender spellings:", raw["gender"].nunique(), "->", sorted(raw["gender"].dropna().unique()))
print("order_status spellings:", raw["order_status"].nunique(), "->", sorted(raw["order_status"].dropna().unique()))

text_cols = ["order_id", "customer_id", "customer_name", "gender", "region", "city",
             "product_category", "product_name", "payment_method", "order_status"]
ws = {c: int((raw[c].dropna() != raw[c].dropna().str.strip()).sum()) for c in text_cols}
print("\nvalues with leading/trailing spaces:", {k: v for k, v in ws.items() if v} or "none")

# Collapse each date into a pattern: digits -> 9, month names -> A
patterns = (raw["order_date"].dropna()
            .str.replace(r"\d", "9", regex=True)
            .str.replace(r"[A-Za-z]+", "A", regex=True)
            .value_counts())
print("\norder_date formats found:")
display(patterns.to_frame("rows"))

gender spellings: 11 -> ['F', 'FEMALE', 'Female', 'M', 'MALE', 'Male', 'Other', 'f', 'female', 'm', 'male']
order_status spellings: 9 -> ['Cancelled', 'Delivered', 'Pending', 'Returned', 'Shipped', 'cancelled', 'delivered', 'pending', 'shipped']

values with leading/trailing spaces: none

order_date formats found:


,rows
order_date,
9999-99-99,4007
99/99/9999,150
"A 99, 9999",123


### What the report found

| Problem | Where | Fixed in |
|---|---|---|
| 30 rows are completely blank | every column | Section 2 |
| 80 exact duplicate records | whole row | Section 2 |
| 11 spellings of gender, 9 of order status (`Delivered` / `delivered`) | `gender`, `order_status` | Section 3 |
| 3 different date formats stored as text | `order_date` | Section 3 |
| Impossible values: ages like -7 / 150 / 999, quantity -1 / 999, negative shipping cost and negative days to ship | `age`, `quantity`, `shipping_cost`, `days_to_ship` | Section 4 |
| Extreme values (e.g. 100 days to ship) | numeric columns | Section 5 |
| Gaps in 5 numeric columns | `age`, `quantity`, `discount_pct`, `customer_satisfaction`, `days_to_ship` | Section 6 |
| Wrong dtypes (dates as text, whole numbers as floats, flag stored as object) | several | Section 7 |

## 2. Empty rows and duplicates

**Decisions**

- **Blank rows are deleted.** A row with every field empty holds no information, so there is nothing to impute.
- **Exact duplicates are deleted, keeping the first copy.** Two identical records of the same order would double-count sales and profit. After removal `order_id` must be unique, which proves no near-duplicates were hiding behind it.

In [30]:
# 2a. fully empty rows
empty_mask = df.isna().all(axis=1)
n_empty = int(empty_mask.sum())
df = df.loc[~empty_mask]
log("Empty rows", "deleted rows where every column was null", n_empty, "no information to recover")

# 2b. exact duplicate rows (keep the first occurrence)
n_dup = int(df.duplicated().sum())
df = df.drop_duplicates(keep="first")
log("Duplicates", "deleted exact duplicate rows, kept first", n_dup, "would double-count sales and profit")

print(f"Blank rows removed:      {n_empty}")
print(f"Duplicate rows removed:  {n_dup}")
print(f"Rows remaining:          {len(df):,}")
print(f"order_id is unique:      {df['order_id'].is_unique}")

Blank rows removed:      30
Duplicate rows removed:  80
Rows remaining:          4,200
order_id is unique:      True


## 3. Standardise text and dates

**Decisions**

- **Gender:** `m`, `M`, `male`, `MALE` become `Male`; `f`, `F`, `female`, `FEMALE` become `Female`. The existing `Other` category is a real value and is kept as is.
- **Order status:** converted to Title Case so `delivered` and `Delivered` are one category.
- **Dates:** the file mixes `2024-08-15`, `15/08/2024` and `August 15, 2024`. Each format is parsed separately with an explicit format string, then merged into one `datetime` column. Slash dates are read as **day/month/year**: the check below shows many of them have a first number above 12 and none has a second number above 12, so month-first is impossible.

In [31]:
# 3a. text columns: trim stray spaces
TEXT_COLS = ["order_id", "customer_id", "customer_name", "gender", "region", "city",
             "product_category", "product_name", "payment_method", "order_status"]
for c in TEXT_COLS:
    df[c] = df[c].str.strip()

# 3b. gender
gender_map = {"m": "Male", "male": "Male", "f": "Female", "female": "Female", "other": "Other"}
gender_clean = df["gender"].str.lower().map(gender_map)
assert gender_clean.notna().all(), "unmapped gender value found"
n_gender = int((df["gender"] != gender_clean).sum())
df["gender"] = gender_clean
log("Standardise", "gender mapped to Male / Female / Other", n_gender, "11 spellings collapsed to 3")

# 3c. order status
status_clean = df["order_status"].str.title()
n_status = int((df["order_status"] != status_clean).sum())
df["order_status"] = status_clean
log("Standardise", "order_status converted to Title Case", n_status, "delivered/Delivered treated as one category")

print("gender:", df["gender"].value_counts().to_dict())
print("order_status:", df["order_status"].value_counts().to_dict())

gender: {'Other': 1419, 'Male': 1408, 'Female': 1373}
order_status: {'Delivered': 2731, 'Returned': 612, 'Cancelled': 346, 'Shipped': 327, 'Pending': 184}


In [32]:
# 3d. dates: confirm day-first for slash dates before parsing
s = df["order_date"]
is_iso = s.str.match(r"^\d{4}-\d{2}-\d{2}$")
is_slash = s.str.match(r"^\d{1,2}/\d{1,2}/\d{4}$")
is_long = ~(is_iso | is_slash)

parts = s[is_slash].str.split("/", expand=True).astype(int)
print(f"Slash dates: {len(parts)} | first number > 12: {(parts[0] > 12).sum()} | second number > 12: {(parts[1] > 12).sum()}")
print("-> second number is never above 12, so the format is day/month/year\n")

order_date_clean = pd.concat([
    pd.to_datetime(s[is_iso],   format="%Y-%m-%d"),
    pd.to_datetime(s[is_slash], format="%d/%m/%Y"),
    pd.to_datetime(s[is_long],  format="%B %d, %Y"),
]).reindex(df.index)

assert order_date_clean.notna().all(), "some dates failed to parse"
df["order_date"] = order_date_clean
log("Standardise", "order_date: 3 text formats parsed into one datetime column",
    int((~is_iso).sum()), "slash dates read as day/month/year (verified)")

print(f"Formats: {int(is_iso.sum())} ISO, {int(is_slash.sum())} dd/mm/yyyy, {int(is_long.sum())} 'Month dd, yyyy'")
print(f"Date range: {df['order_date'].min().date()} to {df['order_date'].max().date()}")
assert df["order_id"].is_unique

Slash dates: 147 | first number > 12: 83 | second number > 12: 0
-> second number is never above 12, so the format is day/month/year

Formats: 3933 ISO, 147 dd/mm/yyyy, 120 'Month dd, yyyy'
Date range: 2020-01-01 to 2024-12-30


## 4. Fix impossible values

Values outside the business rules from section 1d cannot be real. They are handled in two ways:

- **Quantity above 100 (the `999` placeholder): delete the row.** `sales_amount` and `profit` on these rows were calculated *from* the bad quantity (sales of up to 17 million for a single order), so the row's money columns are wrong too and cannot be rebuilt reliably. Keeping them would wreck every total and average.
- **Every other impossible value: set to missing (`NaN`).** Negative ages, ages of 150 or 999, negative quantities, negative shipping costs and negative shipping days are individual bad cells. The rest of each row is fine, so the row stays and the cell is filled in section 6.

In [33]:
# 4a. quantity placeholder (999): delete the whole row
sentinel = df["quantity"] > RANGE_RULES["quantity"][1]
n_sentinel = int(sentinel.sum())
df = df.loc[~sentinel]
log("Impossible values", "deleted rows with quantity = 999 placeholder", n_sentinel,
    "sales_amount/profit inflated by the bad quantity (up to 17M)")
print(f"Rows deleted (quantity placeholder): {n_sentinel}")

# 4b. every other out-of-range value -> NaN (filled later in section 6)
for col in RANGE_RULES:
    bad = out_of_range(df, col)
    n_bad = int(bad.sum())
    if n_bad:
        df.loc[bad, col] = np.nan
        log("Impossible values", f"{col}: out-of-range values set to missing", n_bad,
            f"outside allowed range {RANGE_RULES[col]}")
    print(f"{col:<22} set to NaN: {n_bad}")

Rows deleted (quantity placeholder): 8
age                    set to NaN: 24
quantity               set to NaN: 18
unit_price             set to NaN: 0
sales_amount           set to NaN: 0
shipping_cost          set to NaN: 2
days_to_ship           set to NaN: 5
discount_pct           set to NaN: 0
customer_satisfaction  set to NaN: 0


## 5. Outlier detection (IQR method)

A value is an outlier when it falls below `Q1 - 1.5 x IQR` or above `Q3 + 1.5 x IQR`, where IQR is `Q3 - Q1`. I chose IQR over Z-score because several columns are heavily skewed, and IQR is not pulled around by the extreme values it is trying to find.

`customer_satisfaction` is skipped because it is a bounded 1 to 5 rating.

In [34]:
OUTLIER_COLS = ["age", "quantity", "unit_price", "discount_pct", "sales_amount",
                "profit", "shipping_cost", "days_to_ship"]

def iqr_bounds(series):
    s = series.dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

rows = []
for col in OUTLIER_COLS:
    lo, hi = iqr_bounds(df[col])
    s = df[col].dropna()
    rows.append({"column": col, "lower_fence": round(lo, 2), "upper_fence": round(hi, 2),
                 "below": int((s < lo).sum()), "above": int((s > hi).sum()),
                 "pct_flagged": round(((s < lo) | (s > hi)).mean() * 100, 2)})
display(pd.DataFrame(rows))

,column,lower_fence,upper_fence,below,above,pct_flagged
0,age,2.00,66.00,2,10,0.30
1,quantity,-4.50,15.50,0,0,0.00
2,unit_price,-17562.00,31088.60,0,466,11.12
3,discount_pct,-0.21,0.62,0,0,0.00
4,sales_amount,-68952.01,121504.81,0,582,13.88
5,profit,-13885.97,25025.26,0,543,12.95
6,shipping_cost,13.61,112.88,7,10,0.41
7,days_to_ship,-4.50,15.50,0,6,0.15


The money columns (`unit_price`, `sales_amount`, `profit`) are flagged heavily, but that is expected: a laptop costs far more than a bag of sugar. The fairer test is to run IQR **within each product category**.

In [35]:
rows = []
for col in ["unit_price", "sales_amount", "profit"]:
    flagged = 0
    for cat, g in df.groupby("product_category"):
        lo, hi = iqr_bounds(g[col])
        flagged += int(((g[col] < lo) | (g[col] > hi)).sum())
    rows.append({"column": col, "flagged_overall": int(((df[col] < iqr_bounds(df[col])[0]) | (df[col] > iqr_bounds(df[col])[1])).sum()),
                 "flagged_within_category": flagged})
display(pd.DataFrame(rows))

,column,flagged_overall,flagged_within_category
0,unit_price,466,0
1,sales_amount,582,130
2,profit,543,170


### Decision for each column

| Column | Decision | Reason |
|---|---|---|
| `age` | **Retain** | The few flagged ages (age 1, and ages 67 to 72) are possible human ages. Capping would overwrite real values. |
| `quantity` | **Retain** | Nothing flagged once the `999` placeholder is gone. |
| `unit_price` | **Retain** | The overall flags come from category price tiers (Electronics vs Groceries). Inside each category nothing is flagged. |
| `discount_pct` | **Retain** | Nothing flagged; values stay within 0 to 0.40. |
| `sales_amount` | **Retain** | Remaining flags look like large orders (high unit price x high quantity) and nothing about them is impossible. Removing them would delete the most valuable orders. |
| `profit` | **Retain** | Same reasoning as `sales_amount`. |
| `shipping_cost` | **Cap** at the IQR fences | A handful of values sit far outside an otherwise bell-shaped spread. Capping keeps the rows and limits the damage to averages. |
| `days_to_ship` | **Remove the value (set to missing, filled in section 6)** | Almost every order ships in 0 to 10 days, so values like 45 and 100 are entry errors, not slow deliveries. Capping would invent a number; treating them as missing is more honest. |

In [36]:
# shipping_cost: cap at the IQR fences
lo, hi = iqr_bounds(df["shipping_cost"])
n_cap = int(((df["shipping_cost"] < lo) | (df["shipping_cost"] > hi)).sum())
df["shipping_cost"] = df["shipping_cost"].clip(lower=lo, upper=hi)
log("Outliers", f"shipping_cost capped to [{lo:.2f}, {hi:.2f}]", n_cap, "keeps rows, limits influence on averages")
print(f"shipping_cost capped: {n_cap}  (fences {lo:.2f} to {hi:.2f})")

# days_to_ship: values above the upper fence -> missing
lo, hi = iqr_bounds(df["days_to_ship"])
extreme = df["days_to_ship"] > hi
n_ext = int(extreme.sum())
df.loc[extreme, "days_to_ship"] = np.nan
log("Outliers", f"days_to_ship above {hi:.1f} set to missing", n_ext, "entry errors (typical range is 0-10 days)")
print(f"days_to_ship set to missing: {n_ext}  (upper fence {hi:.1f})")

log("Outliers", "age, quantity, unit_price, discount_pct, sales_amount, profit retained", 0,
    "flagged values are plausible (category price tiers, genuine large orders, real ages)")

shipping_cost capped: 17  (fences 13.61 to 112.88)
days_to_ship set to missing: 6  (upper fence 15.5)


## 6. Missing data handling

| Column | Strategy | Why |
|---|---|---|
| `age` | **Median** | Roughly symmetric, but the median is not affected by the few unusual ages that remain. Rounded to a whole number. |
| `quantity` | **Median** | Whole-number counts; the median is always a realistic order size. |
| `discount_pct` | **Median** | Discounts are spread across 0 to 40%, so a missing discount is not safely "0%". The median is a neutral middle value. |
| `days_to_ship` | **Median** | Typical shipping time is the best neutral guess; rounded to whole days. |
| `shipping_cost` | **Median** | Only the cells made missing in section 4. Median avoids pulling toward the capped extremes. |
| `customer_satisfaction` | **Leave as missing, add an indicator column** | About 8% of orders have no rating, most likely because the customer did not respond. Filling in a score would invent opinions and shift the average. Analysts can still ignore blanks (pandas does by default), and the new `satisfaction_missing` flag records which orders were not rated. |
| all other columns | **Nothing needed** | They have no gaps once the blank rows were removed. |

In [37]:
gaps = df.isna().sum()
print("Missing values going into this step:")
display(gaps[gaps > 0].to_frame("nulls"))

# whole-number columns: median rounded to an integer
for col in ["age", "quantity", "days_to_ship"]:
    n = int(df[col].isna().sum())
    fill = int(round(df[col].median()))
    df[col] = df[col].fillna(fill)
    log("Missing data", f"{col}: filled with median ({fill})", n, "robust to unusual values")
    print(f"{col:<15} filled {n:>4} with median {fill}")

# decimal columns: plain median
for col in ["discount_pct", "shipping_cost"]:
    n = int(df[col].isna().sum())
    fill = round(float(df[col].median()), 4)
    df[col] = df[col].fillna(fill)
    log("Missing data", f"{col}: filled with median ({fill})", n, "robust to unusual values")
    print(f"{col:<15} filled {n:>4} with median {fill}")

# satisfaction: keep the gaps, record where they are
df["satisfaction_missing"] = df["customer_satisfaction"].isna()
n_sat = int(df["satisfaction_missing"].sum())
log("Missing data", "customer_satisfaction left missing; added satisfaction_missing flag", n_sat,
    "imputing a rating would invent customer opinions")
print(f"customer_satisfaction left missing: {n_sat} (flagged in satisfaction_missing)")

remaining = df.isna().sum()
assert remaining.drop("customer_satisfaction").sum() == 0

Missing values going into this step:


,nulls
age,149
quantity,126
discount_pct,135
shipping_cost,2
customer_satisfaction,345
days_to_ship,108


age             filled  149 with median 34
quantity        filled  126 with median 5
days_to_ship    filled  108 with median 6
discount_pct    filled  135 with median 0.21
shipping_cost   filled    2 with median 63.46
customer_satisfaction left missing: 345 (flagged in satisfaction_missing)


## 7. Data type correction

| Column(s) | Target dtype | Why |
|---|---|---|
| `order_id`, `customer_id`, `customer_name`, text categories | `string` | Identifiers are labels, not numbers. |
| `order_date` | `datetime64` | Already converted in section 3. |
| `age`, `quantity`, `days_to_ship`, `customer_satisfaction` | `Int64` | Whole numbers. pandas' nullable `Int64` also allows the intentional blanks in `customer_satisfaction`. |
| `unit_price`, `discount_pct`, `sales_amount`, `profit`, `shipping_cost` | `float64` | Monetary and ratio values. |
| `return_flag` | `bool` | True / False flag. |

In [38]:
for col in TEXT_COLS:
    df[col] = df[col].astype("string")

for col in ["age", "quantity", "days_to_ship", "customer_satisfaction"]:
    df[col] = df[col].astype("Int64")

for col in ["unit_price", "discount_pct", "sales_amount", "profit", "shipping_cost"]:
    df[col] = df[col].astype("float64")

flag = df["return_flag"].map({True: True, False: False, "True": True, "False": False})
assert flag.notna().all()
df["return_flag"] = flag.astype(bool)

log("Data types", "converted IDs/text to string, counts to Int64, money to float64, return_flag to bool", len(EXPECTED_DTYPES),
    "order_date already datetime from section 3")

dtype_after = dtype_report(df)
display(dtype_after)
assert dtype_after["correct"].all()

,column,current_dtype,expected,correct
0,order_id,string,string,True
1,order_date,datetime64[us],datetime,True
2,customer_id,string,string,True
3,customer_name,string,string,True
4,age,Int64,integer,True
5,gender,string,string,True
6,region,string,string,True
7,city,string,string,True
8,product_category,string,string,True
9,product_name,string,string,True


### Final validation
A few assertions to prove the clean data obeys the rules.

In [39]:
assert df["order_id"].is_unique
assert df.duplicated().sum() == 0
assert df[list(EXPECTED_DTYPES)].drop(columns="customer_satisfaction").isna().sum().sum() == 0
for col in RANGE_RULES:
    assert not out_of_range(df, col).any(), f"{col} still has out-of-range values"
assert set(df["gender"]) == {"Male", "Female", "Other"}
assert df["order_status"].str.istitle().all()

# Returned orders and the return flag should agree
print(pd.crosstab(df["order_status"], df["return_flag"]))
print("\nAll checks passed.")

return_flag   False  True 
order_status              
Cancelled       346      0
Delivered      2725      0
Pending         184      0
Returned          0    610
Shipped         327      0

All checks passed.


## 8. Before vs. after

In [40]:
after = snapshot(df)
summary = pd.DataFrame({"Before (raw)": before, "After (clean)": after})
display(summary)

,Before (raw),After (clean)
Row count,4310,4192
Null cells (total),1455,345
Columns containing nulls,21,1
Duplicate rows,109,0
Columns with correct dtype,15 / 21,21 / 21
Dtype accuracy,71%,100%


### Per-column detail

In [41]:
cols = list(EXPECTED_DTYPES)
detail = pd.DataFrame({
    "nulls_before": raw[cols].isna().sum(),
    "nulls_after": df[cols].isna().sum(),
    "dtype_before": raw[cols].dtypes.astype(str),
    "dtype_after": df[cols].dtypes.astype(str),
    "correct_before": dtype_report(raw).set_index("column")["correct"],
    "correct_after": dtype_report(df).set_index("column")["correct"],
})
display(detail)

,nulls_before,nulls_after,dtype_before,dtype_after,correct_before,correct_after
order_id,30,0,str,string,True,True
order_date,30,0,str,datetime64[us],False,True
customer_id,30,0,str,string,True,True
customer_name,30,0,str,string,True,True
age,160,0,float64,Int64,False,True
gender,30,0,str,string,True,True
region,30,0,str,string,True,True
city,30,0,str,string,True,True
product_category,30,0,str,string,True,True
product_name,30,0,str,string,True,True


## 9. Save the cleaned dataset

In [42]:
df.to_csv(CLEAN_PATH, index=False, date_format="%Y-%m-%d")

check = pd.read_csv(CLEAN_PATH, parse_dates=["order_date"])
print(f"Saved {CLEAN_PATH}: {check.shape[0]:,} rows x {check.shape[1]} columns")
print("order_date range after reload:", check["order_date"].min().date(), "to", check["order_date"].max().date())

Saved retail_sales_cleaned.csv: 4,192 rows x 22 columns
order_date range after reload: 2020-01-01 to 2024-12-30


## Cleaning log
Every action taken, in order.

In [43]:
log_df = pd.DataFrame(cleaning_log)
log_df.index = log_df.index + 1
display(log_df)

,step,action,affected,why
1,Empty rows,deleted rows where every column was null,30,no information to recover
2,Duplicates,"deleted exact duplicate rows, kept first",80,would double-count sales and profit
3,Standardise,gender mapped to Male / Female / Other,431,11 spellings collapsed to 3
4,Standardise,order_status converted to Title Case,152,delivered/Delivered treated as one category
5,Standardise,order_date: 3 text formats parsed into one dat...,267,slash dates read as day/month/year (verified)
6,Impossible values,deleted rows with quantity = 999 placeholder,8,sales_amount/profit inflated by the bad quanti...
7,Impossible values,age: out-of-range values set to missing,24,"outside allowed range (1, 100)"
8,Impossible values,quantity: out-of-range values set to missing,18,"outside allowed range (1, 100)"
9,Impossible values,shipping_cost: out-of-range values set to missing,2,"outside allowed range (0, None)"
10,Impossible values,days_to_ship: out-of-range values set to missing,5,"outside allowed range (0, None)"


## Summary of decisions

- **Deleted:** blank rows, exact duplicates, and rows carrying the `999` quantity placeholder (their sales and profit were inflated by the bad value).
- **Standardised:** gender (3 values), order status (Title Case), and dates (one `datetime` column; slash dates verified as day/month/year).
- **Impossible values:** turned into missing cells, then filled with the column median.
- **Outliers:** `shipping_cost` capped; `days_to_ship` entry errors treated as missing; price, sales, profit, age, quantity and discount retained because their extremes are legitimate.
- **Left missing on purpose:** `customer_satisfaction`, with a `satisfaction_missing` flag, because a rating cannot be guessed.
- **Result:** analysis-ready file `retail_sales_cleaned.csv`.